In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
pd.set_option('display.max_columns',30)
pd.set_option('display.max_rows',500)


In [3]:
df=pd.read_csv('../data/raw/cc_defaults.csv')
# display(df.head())
# display(df.info())

In [ ]:
# 1. Statistical summary of all columns
print("=== STATISTICAL SUMMARY ===")
display(df.describe())

# 2. Check for missing values
print("\n=== MISSING VALUES PER COLUMN ===")
display(df.isnull().sum())

# 3. Target Variable Analysis
# Automatically find the target column (usually 'default.payment.next.month')
target_col = [col for col in df.columns if 'default' in col.lower()][0]

print(f"\n=== TARGET COLUMN: {target_col} ===")
print("Raw Counts:")
display(df[target_col].value_counts())

print("\nPercentages:")
display(df[target_col].value_counts(normalize=True) * 100)

In [ ]:
# Create a figure with two subplots side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: Target Distribution
sns.countplot(data=df, x=target_col, ax=axes[0], palette='viridis')
axes[0].set_title('Distribution of Defaults (0 = No, 1 = Yes)')
axes[0].set_xlabel('Default Payment Next Month')
axes[0].set_ylabel('Number of Clients')

# Chart 2: Correlation Heatmap
# We'll calculate the correlation matrix for all columns
corr_matrix = df.corr()

# Plotting the heatmap
sns.heatmap(corr_matrix, ax=axes[1], cmap='coolwarm', annot=False, fmt=".1f", linewidths=.5, vmin=-1, vmax=1, center=0)
axes[1].set_title('Feature Correlation Heatmap')

# Display the plots
plt.tight_layout()
plt.show()

In [4]:
df['EDUCATION']=df['EDUCATION'].replace([0,5,6],4)

In [5]:
df['MARRIAGE']=df['MARRIAGE'].replace(0,3)

In [6]:
from sklearn.model_selection import train_test_split

# 1. Drop the ID column
df_clean = df.drop('ID', axis=1)

# 2. Separate X (Features) and y (Target)
X = df_clean.drop('default.payment.next.month', axis=1)
y = df_clean['default.payment.next.month']

# 3. First Split: Carve out 20% for the final Test Set
# This leaves 80% in "temp" (which we will split again)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Second Split: Split the remaining 80% into Train and Validation
# If we want 20% of the TOTAL data for validation, that is exactly 25% (0.25) of the 80% we have left.
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print("=== 3-WAY DATA SPLIT SUCCESSFUL ===")
print(f"X_train shape (Training):   {X_train.shape} - 60%")
print(f"X_val shape   (Validation): {X_val.shape} - 20%")
print(f"X_test shape  (Testing):    {X_test.shape} - 20%")

=== 3-WAY DATA SPLIT SUCCESSFUL ===
X_train shape (Training):   (18000, 23) - 60%
X_val shape   (Validation): (6000, 23) - 20%
X_test shape  (Testing):    (6000, 23) - 20%


In [7]:
# Print the original row numbers for the first 5 rows in X_train
print("X_train row numbers:")
print(X_train.head().index)

# Print the original row numbers for the first 5 rows in y_train
print("\ny_train row numbers:")
print(y_train.head().index)

X_train row numbers:
Index([10574, 11330, 7127, 25150, 6000], dtype='int64')

y_train row numbers:
Index([10574, 11330, 7127, 25150, 6000], dtype='int64')


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Initialize the Random Forest (We will use 100 decision trees)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. Train the model (This is where the AI actually learns the patterns!)
print("Training the Random Forest...")
rf_model.fit(X_train, y_train)

# 3. Give the AI its first test using the Validation data
y_val_pred = rf_model.predict(X_val)


from sklearn.metrics import classification_report, confusion_matrix

# Print the detailed report
print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_val, y_val_pred))

# Print the raw matrix numbers
print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_val, y_val_pred))

# 4. Grade the test
val_accuracy = accuracy_score(y_val, y_val_pred)

print("=== AI TRAINING COMPLETE ===")
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")

Training the Random Forest...
=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0       0.84      0.95      0.89      4673
           1       0.66      0.35      0.46      1327

    accuracy                           0.82      6000
   macro avg       0.75      0.65      0.67      6000
weighted avg       0.80      0.82      0.79      6000


=== CONFUSION MATRIX ===
[[4430  243]
 [ 863  464]]
=== AI TRAINING COMPLETE ===
Validation Accuracy: 81.57%


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Re-initialize the model with class_weight='balanced'
rf_balanced = RandomForestClassifier(
    n_estimators=100, 
    class_weight='balanced',  # <--- THIS IS THE MAGIC KNOB
    random_state=42
)

# 2. Retrain on training data
rf_balanced.fit(X_train, y_train)

# 3. Predict on Validation data
y_val_pred_balanced = rf_balanced.predict(X_val)

# 4. Print results
print("=== BALANCED RANDOM FOREST REPORT ===")
print(classification_report(y_val, y_val_pred_balanced))

print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_val, y_val_pred_balanced))

=== BALANCED RANDOM FOREST REPORT ===
              precision    recall  f1-score   support

           0       0.83      0.95      0.89      4673
           1       0.66      0.33      0.44      1327

    accuracy                           0.81      6000
   macro avg       0.74      0.64      0.66      6000
weighted avg       0.79      0.81      0.79      6000


=== CONFUSION MATRIX ===
[[4446  227]
 [ 892  435]]


In [10]:
import numpy as np

# 1. Ask the AI for the PROBABILITY of default, not just the final vote
# (This returns a list of percentages, like 0.15, 0.82, 0.31...)
probabilities = rf_balanced.predict_proba(X_val)[:, 1]

# 2. OVERRIDE: Set our own strict banking threshold at 30%
threshold = 0.30
y_val_pred_custom = (probabilities >= threshold).astype(int)

# 3. Print the new results
print(f"=== RESULTS WITH {threshold*100}% THRESHOLD ===")
print(classification_report(y_val, y_val_pred_custom))

print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_val, y_val_pred_custom))

=== RESULTS WITH 30.0% THRESHOLD ===
              precision    recall  f1-score   support

           0       0.87      0.85      0.86      4673
           1       0.51      0.54      0.52      1327

    accuracy                           0.78      6000
   macro avg       0.69      0.70      0.69      6000
weighted avg       0.79      0.78      0.79      6000


=== CONFUSION MATRIX ===
[[3992  681]
 [ 616  711]]


In [11]:
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

print("1. Generating synthetic defaulters to balance the data...")
smote = SMOTE(random_state=42)

# We ONLY apply SMOTE to the training data!
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("2. Training a fresh AI on the new 50/50 dataset...")
rf_smote = RandomForestClassifier(n_estimators=100, random_state=42)
rf_smote.fit(X_train_smote, y_train_smote)

print("3. Testing the AI on the pristine Validation data...")
y_val_pred_smote = rf_smote.predict(X_val)

print("\n=== SMOTE RANDOM FOREST REPORT ===")
print(classification_report(y_val, y_val_pred_smote))

print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_val, y_val_pred_smote))

import numpy as np

# 1. Ask the AI for the PROBABILITY of default, not just the final vote
# (This returns a list of percentages, like 0.15, 0.82, 0.31...)
probabilities = rf_smote.predict_proba(X_val)[:, 1]

# 2. OVERRIDE: Set our own strict banking threshold at 30%
threshold = 0.30
y_val_pred_smote_custom = (probabilities >= threshold).astype(int)

# 3. Print the new results
print(f"=== SMOTE RANDOM FOREST REPORT RESULTS WITH {threshold*100}% THRESHOLD ===")
print(classification_report(y_val, y_val_pred_smote_custom))

print("\n=== SMOTE RANDOM FOREST CONFUSION MATRIX ===")
print(confusion_matrix(y_val, y_val_pred_smote_custom))

1. Generating synthetic defaulters to balance the data...
2. Training a fresh AI on the new 50/50 dataset...
3. Testing the AI on the pristine Validation data...

=== SMOTE RANDOM FOREST REPORT ===
              precision    recall  f1-score   support

           0       0.86      0.88      0.87      4673
           1       0.53      0.49      0.51      1327

    accuracy                           0.79      6000
   macro avg       0.69      0.68      0.69      6000
weighted avg       0.78      0.79      0.79      6000


=== CONFUSION MATRIX ===
[[4090  583]
 [ 680  647]]
=== SMOTE RANDOM FOREST REPORT RESULTS WITH 30.0% THRESHOLD ===
              precision    recall  f1-score   support

           0       0.89      0.63      0.74      4673
           1       0.36      0.73      0.48      1327

    accuracy                           0.65      6000
   macro avg       0.63      0.68      0.61      6000
weighted avg       0.78      0.65      0.68      6000


=== SMOTE RANDOM FOREST CONFUS

In [ ]:
print("\n=== SMOTE RANDOM FOREST REPORT ===")
print(classification_report(y_val, y_val_pred_smote))

print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_val, y_val_pred_smote))


print(f"\n=== SMOTE RANDOM FOREST REPORT RESULTS WITH {threshold*100}% THRESHOLD ===")
print(classification_report(y_val, y_val_pred_smote_custom))

print("\n=== SMOTE RANDOM FOREST CONFUSION MATRIX ===")
print(confusion_matrix(y_val, y_val_pred_smote_custom))


=== SMOTE RANDOM FOREST REPORT ===
              precision    recall  f1-score   support

           0       0.86      0.88      0.87      4673
           1       0.53      0.49      0.51      1327

    accuracy                           0.79      6000
   macro avg       0.69      0.68      0.69      6000
weighted avg       0.78      0.79      0.79      6000


=== CONFUSION MATRIX ===
[[4090  583]
 [ 680  647]]

=== SMOTE RANDOM FOREST REPORT RESULTS WITH 30.0% THRESHOLD ===
              precision    recall  f1-score   support

           0       0.89      0.63      0.74      4673
           1       0.36      0.73      0.48      1327

    accuracy                           0.65      6000
   macro avg       0.63      0.68      0.61      6000
weighted avg       0.78      0.65      0.68      6000


=== SMOTE RANDOM FOREST CONFUSION MATRIX ===
[[2948 1725]
 [ 352  975]]


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ID                          30000 non-null  int64  
 1   LIMIT_BAL                   30000 non-null  float64
 2   SEX                         30000 non-null  int64  
 3   EDUCATION                   30000 non-null  int64  
 4   MARRIAGE                    30000 non-null  int64  
 5   AGE                         30000 non-null  int64  
 6   PAY_0                       30000 non-null  int64  
 7   PAY_2                       30000 non-null  int64  
 8   PAY_3                       30000 non-null  int64  
 9   PAY_4                       30000 non-null  int64  
 10  PAY_5                       30000 non-null  int64  
 11  PAY_6                       30000 non-null  int64  
 12  BILL_AMT1                   30000 non-null  float64
 13  BILL_AMT2                   300

In [15]:
display(df['PAY_0'].value_counts())

PAY_0
 0    14737
-1     5686
 1     3688
-2     2759
 2     2667
 3      322
 4       76
 5       26
 8       19
 6       11
 7        9
Name: count, dtype: int64